In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

import pickle

# Generate hotspots of unfairness based on movement patterns

#### Read the Atlanta's 2020 US Census blocks and the stop segments

In [ ]:
path_atlanta_blocks = './experiments/atlanta_census_blocks.zip'
atlanta_blocks = gpd.read_file(path_atlanta_blocks)['geometry'].to_crs("EPSG:4326").to_frame()
display(atlanta_blocks)


path_stop_df = './data_simulator/huge_dataset/dataset_simulator_trajectories.compressed.parquet.stops.parquet'
stop_df = pd.read_parquet(path_stop_df)
stop_df = gpd.GeoDataFrame(stop_df, 
                           geometry=gpd.points_from_xy(stop_df.lng, stop_df.lat), 
                           crs="EPSG:4326").loc[:, ['uid', 'geometry']]
display(stop_df)

#### Filter out the blocks that do not contain any stop segment.

In [ ]:
# Associated each stop segment to a census block via its centroid.
# This will be useful when building hotspots made of multiple separate regions: we can use the candidate
# generation algorithm to see where there are objects associated with more than 1 separate region, and use them
# as seeds to build this kind of hotspots.
mapped_stops = stop_df.sjoin(atlanta_blocks,
                             how="left",
                             predicate="within")[['uid', 'index_right']]

# Filter out the blocks that do not contain any stop segment.
list_nonempty_blocks = mapped_stops['index_right'].unique()
sel_atlanta_blocks = atlanta_blocks.loc[list_nonempty_blocks].copy()
display(sel_atlanta_blocks.shape)
# sel_atlanta_blocks.plot()


# Remove some dataframes from memory (no more necessary for here on).
del atlanta_blocks, mapped_stops

### Read the synthetic unfair labels from disk

In [ ]:
path_unfair_dataset = './experiments/unfair_datasets.pkl'
with open(path_unfair_dataset, "rb") as f:
    datasets = pickle.load(f)

In [ ]:
tuple_idx_mps = 0
tuple_idx_list_objs = 1
tuple_idx_labels = 2

idx_dataset = 226
idx_multipolygon = 0

num_hotspots = len(datasets['data'][idx_dataset][tuple_idx_mps])
multipolygon = datasets['data'][idx_dataset][tuple_idx_mps][idx_multipolygon]
list_object_ids = datasets['data'][idx_dataset][tuple_idx_list_objs][idx_multipolygon]
labels = datasets['data'][idx_dataset][tuple_idx_labels]

In [ ]:
print(num_hotspots)
display(list_object_ids)
uid_random = np.random.choice(list_object_ids)
print(f"Selected user {uid_random}")
sel_uid = stop_df.loc[stop_df['uid'] == uid_random]
display(sel_uid)

**DEBUG**: plot a simple Folium map of the Atlanta's tracts -- nonempty vs empty.

In [ ]:
import folium

# Base map centered on candidates
minx, miny, maxx, maxy = sel_atlanta_blocks.total_bounds
m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=12, prefer_canvas=True)

def style_fn1(feature):
    return {
        "color": "blue",
        "weight": 0.5,
        "fillColor": "red",
        "fillOpacity": 0.3,
    }
folium.GeoJson(sel_atlanta_blocks, style_fn1).add_to(m)

#def style_fn2(feature):
#    return {
#        "fillColor": "blue",
#        "fillOpacity": 0.7,
#    }
#folium.GeoJson(multipolygon, style_fn2).add_to(m)


# Plot the stop centroids of a specific user.
#def style_fn3(feature):
#    return {
#        "radius": 4,
#        "color": "yellow",
#        "fillColor": "yellow",
#        "fillOpacity": 0.7,
#        "weight": 1,
#    }
#folium.GeoJson(sel_uid, 
#               style_function=style_fn3,
#               marker=folium.CircleMarker()).add_to(m)

m